![Noteable.ac.uk Banner](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/Images/Noteable%20NB%20Header%20Banner.png)

# Interactive Audio Intelligence: From Spectrogram Features to Deep Learning

## Legend

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>blue</b>, the <b>instructions</b> and <b>goals</b> are highlighted. This tells you what we are trying to achieve.
</div>
<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>green</b>, key <b>information</b> and <b>concept explanations</b> are highlighted.
</div>
<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>yellow</b>, <b>exercises</b> and <b>tasks</b> are highlighted for you to try yourself.
</div>
<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>red</b>, <b>error interpretation</b> and <b>debugging tips</b> are highlighted.
</div>

Welcome! This notebook is about **audio machine learning** in a very hands-on way.

Instead of starting with text, we will start with **sound itself**:
- first as a **waveform**
- then as a **spectrogram-style representation**
- then as input for **machine learning models**

Our learning journey is:
1. inspect audio clips visually
2. transform them into mel-spectrograms
3. build a **classical baseline** from compact audio features
4. build a **simple neural audio model** in PyTorch
5. compare the results side by side
6. explore predictions interactively

We will try to use a small spoken-audio dataset through `torchaudio`. If that is unavailable, the notebook includes a lightweight generated-audio fallback so the complete workflow still runs smoothly in class.

## 1. Setting Up Our Toolkit

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Import the libraries we need for audio loading, spectrogram creation, machine learning, neural modelling, visualisation, and learner interaction.
</div>

Click the code cell below and press the **<code>▶</code> Run** button in the toolbar, or use <code>Shift + Enter</code>. The print messages will help you follow our progress.

In [ ]:
print("Starting setup: importing audio, machine learning, and visualisation libraries...")

import math
import random
import copy
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

import ipywidgets as widgets
from IPython.display import display, clear_output, Audio

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Subset
import torchaudio

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from sklearn.decomposition import PCA

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


print(f"PyTorch version: {torch.__version__}")
print(f"torchaudio version: {torchaudio.__version__}")
print(f"Computation device: {device}")
import plotly.io as pio
pio.renderers.default = "iframe_connected"
print("Plotly renderer:", pio.renderers.default)
print("Success! Our toolkit is ready.")

## 2. Loading a Lightweight Audio Dataset

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Load a small, classroom-friendly audio dataset. We will try a compact spoken-audio dataset first, and if that is not available, we will switch to a fully runnable generated-audio fallback.
</div>

That way, the notebook stays practical and robust while still showing a genuine audio intelligence workflow.

In [ ]:
print("Loading audio data...")

AUDIO_ROOT = "./audio_intelligence_data"
waveforms = []
records = []

def generate_chirp(start_freq, end_freq, duration, sample_rate, noise_level=0.02):
    num_samples = int(duration * sample_rate)
    
    # Calculate frequencies and cumulative phase using rock-solid NumPy calculations
    freqs = np.linspace(start_freq, end_freq, num_samples)
    phase = 2 * np.pi * np.cumsum(freqs / sample_rate)
    
    # Structural wave configuration
    waveform_np = 0.6 * np.sin(phase) + 0.25 * np.sin(2 * phase)
    envelope = np.sin(np.linspace(0, math.pi, num_samples)) ** 1.2
    noise = noise_level * np.random.randn(num_samples)
    
    final_waveform = waveform_np * envelope + noise
    
    # Normalize securely to protect against clipping or silent zero divisions
    max_val = np.max(np.abs(final_waveform))
    if max_val > 1e-6:
        final_waveform = final_waveform / max_val
        
    return torch.from_numpy(final_waveform).float()

try:
    yesno_dataset = torchaudio.datasets.YESNO(root=AUDIO_ROOT, download=True)
    dataset_source = "torchaudio YESNO spoken-audio dataset"
    for idx in range(len(yesno_dataset)):
        waveform, sample_rate, labels = yesno_dataset[idx]
        waveform = waveform.squeeze(0).float()
        label_id = int(labels[0])
        label_name = "Starts with YES" if label_id == 1 else "Starts with NO"
        label_short = "yes" if label_id == 1 else "no"
        description = "Sequence: " + " ".join(["yes" if value == 1 else "no" for value in labels])
        records.append(
            {
                "sample_id": idx,
                "label_id": label_id,
                "label_name": label_name,
                "label_short": label_short,
                "sample_rate": int(sample_rate),
                "num_samples": int(waveform.numel()),
                "duration_sec": float(waveform.numel() / sample_rate),
                "description": description,
            }
        )
        waveforms.append(waveform)
    print(f"Loaded {len(records)} spoken audio clips from the {dataset_source}.")
except Exception as err:
    print("The spoken-audio dataset was unavailable, so a generated fallback dataset will be created instead.")
    print(f"Download detail: {type(err).__name__}")
    dataset_source = "generated chirp dataset fallback"
    sample_rate = 16000
    clips_per_class = 60
    for idx in range(clips_per_class):
        duration = random.uniform(0.75, 1.15)
        falling = generate_chirp(
            start_freq=random.uniform(800, 1000),
            end_freq=random.uniform(180, 320),
            duration=duration,
            sample_rate=sample_rate,
            noise_level=0.025,
        )
        rising = generate_chirp(
            start_freq=random.uniform(180, 320),
            end_freq=random.uniform(800, 1000),
            duration=duration,
            sample_rate=sample_rate,
            noise_level=0.025,
        )
        records.append(
            {
                "sample_id": len(records),
                "label_id": 0,
                "label_name": "Falling chirp",
                "label_short": "falling",
                "sample_rate": sample_rate,
                "num_samples": int(falling.numel()),
                "duration_sec": float(falling.numel() / sample_rate),
                "description": "Generated downward-frequency chirp",
            }
        )
        waveforms.append(falling)
        records.append(
            {
                "sample_id": len(records),
                "label_id": 1,
                "label_name": "Rising chirp",
                "label_short": "rising",
                "sample_rate": sample_rate,
                "num_samples": int(rising.numel()),
                "duration_sec": float(rising.numel() / sample_rate),
                "description": "Generated upward-frequency chirp",
            }
        )
        waveforms.append(rising)
    print(f"Created {len(records)} generated audio clips for a fully runnable fallback workflow.")

meta_df = pd.DataFrame(records)
id_to_display = meta_df.drop_duplicates("label_id").sort_values("label_id").set_index("label_id")["label_name"].to_dict()
class_names = [id_to_display[idx] for idx in sorted(id_to_display)]

print(f"Dataset source: {dataset_source}")
print(f"Total number of audio clips: {len(meta_df)}")
display(meta_df.head(6))

## 3. First Look at the Audio Data

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Why this matters:</b> Before we train any model, we should understand the dataset. Are the classes balanced? Are clips roughly the same length? These checks help us make better modelling decisions later.
</div>

Run the next cell to inspect the class distribution and clip durations.

In [ ]:
print("Inspecting class balance and clip lengths...")

count_df = meta_df["label_name"].value_counts().rename_axis("label_name").reset_index(name="count")

fig_balance = px.bar(
    count_df,
    x="label_name",
    y="count",
    color="label_name",
    text="count",
    title="Class balance in the audio dataset",
    template="plotly_white",
)
fig_balance.update_traces(textposition="outside")
fig_balance.update_layout(showlegend=False, xaxis_title="Class", yaxis_title="Number of clips")
fig_balance.show(renderer="notebook")

fig_duration = px.histogram(
    meta_df,
    x="duration_sec",
    color="label_name",
    nbins=20,
    opacity=0.8,
    marginal="box",
    title="Clip duration distribution",
    template="plotly_white",
)
fig_duration.update_layout(xaxis_title="Duration in seconds", yaxis_title="Number of clips")
fig_duration.show(renderer="notebook")

summary_df = meta_df.groupby("label_name")[["duration_sec", "num_samples"]].agg(["count", "mean", "median"]).round(3)
display(summary_df)

## 4. Seeing Sound: Waveforms and Mel-Spectrograms

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Convert raw audio into visual representations that machine learning models can use. A waveform shows changes in amplitude over time. A mel-spectrogram shows how energy is distributed across frequency bands over time.
</div>

This is one of the big ideas in audio ML: we often do not feed raw sound directly into a model. Instead, we transform it into a representation that makes useful structure easier to detect.

In [ ]:
print("Preparing audio transforms and visualisation helpers...")

TARGET_SAMPLE_RATE = int(meta_df["sample_rate"].mode().iloc[0])

# EDIT THIS VALUE: number of mel frequency bands.
N_MELS = 32
# EDIT THIS VALUE: FFT size for the spectrogram.
N_FFT = 512
# EDIT THIS VALUE: hop length between analysis windows.
HOP_LENGTH = 128
# EDIT THIS VALUE: fixed time-frame width for the neural model.
TARGET_FRAMES = 96

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
)
db_transform = torchaudio.transforms.AmplitudeToDB(top_db=80)

def resample_if_needed(waveform, original_sr):
    if int(original_sr) == TARGET_SAMPLE_RATE:
        return waveform
    return torchaudio.functional.resample(waveform, int(original_sr), TARGET_SAMPLE_RATE)

def waveform_to_mel_db(waveform, sample_rate):
    wave = resample_if_needed(waveform, sample_rate)
    mel = mel_transform(wave.unsqueeze(0))
    return db_transform(mel).squeeze(0)

def pad_or_trim_spec(spec, target_frames=TARGET_FRAMES):
    if spec.shape[1] < target_frames:
        pad_amount = target_frames - spec.shape[1]
        spec = torch.nn.functional.pad(spec, (0, pad_amount))
    else:
        spec = spec[:, :target_frames]
    return spec

def make_waveform_figure(waveform, sample_rate, title):
    time_axis = np.arange(len(waveform)) / sample_rate
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=time_axis,
            y=waveform.detach().cpu().numpy(),
            mode="lines",
            line=dict(color="#4C72B0"),
            name="Waveform",
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title="Time (seconds)",
        yaxis_title="Amplitude",
        height=320,
        template="plotly_white",
        showlegend=False,
    )
    return fig

def make_spectrogram_figure(mel_db, title):
    fig = px.imshow(
        mel_db.detach().cpu().numpy(),
        aspect="auto",
        origin="lower",
        color_continuous_scale="Viridis",
        labels={"x": "Time frame", "y": "Mel bin", "color": "dB"},
        title=title,
    )
    fig.update_layout(height=420, template="plotly_white")
    return fig

print("Audio transforms are ready.")

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Exercise:</b> In the next cell, change <code>EXAMPLE_INDEX</code> to inspect a different clip. Then run the cell again with <code>Shift + Enter</code> and compare how the waveform and spectrogram change.
</div>

In [ ]:
print("Selecting one audio clip for close inspection...")

import plotly.io as pio
pio.renderers.default = "iframe_connected"

# EDIT THIS VALUE: choose any sample index from 0 to len(meta_df) - 1
EXAMPLE_INDEX = 1

example_row = meta_df.iloc[EXAMPLE_INDEX]
example_waveform = waveforms[EXAMPLE_INDEX]
example_sample_rate = int(example_row["sample_rate"])

# Make sure waveform is 1D, finite, and long enough
example_waveform = torch.as_tensor(example_waveform, dtype=torch.float32)

if example_waveform.ndim > 1:
    example_waveform = example_waveform.mean(dim=0)

example_waveform = torch.nan_to_num(example_waveform, nan=0.0, posinf=0.0, neginf=0.0)

if example_waveform.numel() < N_FFT:
    example_waveform = torch.nn.functional.pad(example_waveform, (0, N_FFT - example_waveform.numel()))

example_mel_db = waveform_to_mel_db(example_waveform, example_sample_rate)
example_mel_db = torch.nan_to_num(example_mel_db, nan=-80.0, posinf=0.0, neginf=-80.0)

print(f"Dataset source: {dataset_source}")
print(f"Sample {EXAMPLE_INDEX} | class: {example_row['label_name']} | duration: {example_row['duration_sec']:.2f} seconds")
print(f"Metadata: {example_row['description']}")
print("Waveform shape:", tuple(example_waveform.shape))
print("Waveform min/max:", float(example_waveform.min()), float(example_waveform.max()))
print("Any NaN in waveform?", bool(torch.isnan(example_waveform).any()))
print("Mel shape:", tuple(example_mel_db.shape))
print("Any NaN in mel?", bool(torch.isnan(example_mel_db).any()))

audio_np = example_waveform.detach().cpu().numpy().astype(np.float32)
display(Audio(data=audio_np, rate=example_sample_rate, autoplay=False))

fig_wave = make_waveform_figure(example_waveform, example_sample_rate, f"Waveform for sample {EXAMPLE_INDEX}")
fig_spec = make_spectrogram_figure(example_mel_db, f"Mel-spectrogram for sample {EXAMPLE_INDEX}")

fig_wave.show(renderer="notebook")
fig_spec.show(renderer="notebook")


## 5. A Classical Audio Baseline with scikit-learn

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Build a classical machine learning baseline using compact engineered audio features. This gives us a strong reference point before we move to a neural model.
</div>

We will extract features such as:
- clip duration
- RMS energy
- peak amplitude
- zero-crossing rate
- average mel-band energy patterns

Then we will train a **Logistic Regression** classifier with `scikit-learn`.

This is an important lesson in ML: a thoughtful baseline is not a formality. It is part of doing rigorous work.

In [ ]:
print("Extracting engineered audio features and training a classical baseline...")

y = meta_df["label_id"].to_numpy()
all_indices = np.arange(len(meta_df))

train_idx, temp_idx = train_test_split(
    all_indices,
    test_size=0.30,
    stratify=y,
    random_state=SEED,
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=y[temp_idx],
    random_state=SEED,
)

meta_df["split"] = "unassigned"
meta_df.loc[train_idx, "split"] = "train"
meta_df.loc[val_idx, "split"] = "validation"
meta_df.loc[test_idx, "split"] = "test"

def zero_crossing_rate(waveform):
    if waveform.numel() < 2:
        return 0.0
    return float(((waveform[:-1] * waveform[1:]) < 0).float().mean().item())

def extract_engineered_features(waveform, sample_rate):
    wave = torch.as_tensor(waveform, dtype=torch.float32)

    if wave.ndim > 1:
        wave = wave.mean(dim=0)

    wave = torch.nan_to_num(wave, nan=0.0, posinf=0.0, neginf=0.0)
    wave = resample_if_needed(wave, sample_rate)

    if wave.numel() < N_FFT:
        wave = torch.nn.functional.pad(wave, (0, N_FFT - wave.numel()))

    duration_sec = max(len(wave) / TARGET_SAMPLE_RATE, 1e-6)
    rms = torch.sqrt(torch.clamp((wave ** 2).mean(), min=1e-12)).item()
    peak = wave.abs().max().item() if wave.numel() else 0.0
    zcr = zero_crossing_rate(wave)

    mel_db = waveform_to_mel_db(wave, TARGET_SAMPLE_RATE)
    mel_db = torch.nan_to_num(mel_db, nan=-80.0, posinf=0.0, neginf=-80.0)

    mel_mean = mel_db.mean(dim=1).detach().cpu().numpy()
    mel_std = mel_db.std(dim=1, correction=0).detach().cpu().numpy()

    low_mel_energy = mel_db[: N_MELS // 2].mean().item()
    high_mel_energy = mel_db[N_MELS // 2 :].mean().item()

    features = np.concatenate(
        [
            np.array([duration_sec, rms, peak, zcr, low_mel_energy, high_mel_energy], dtype=np.float32),
            mel_mean.astype(np.float32),
            mel_std.astype(np.float32),
        ]
    )

    features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
    return features


feature_names = (
    ["duration_sec", "rms", "peak", "zero_crossing_rate", "low_mel_energy", "high_mel_energy"]
    + [f"mel_mean_{i}" for i in range(N_MELS)]
    + [f"mel_std_{i}" for i in range(N_MELS)]
)

X = np.vstack([
    extract_engineered_features(waveform, sample_rate)
    for waveform, sample_rate in zip(waveforms, meta_df["sample_rate"])
])

def compute_metrics(y_true, y_pred, model_name, split_name):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    return {
        "model": model_name,
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

classical_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=SEED)
)

classical_model.fit(X[train_idx], y[train_idx])
classical_val_pred = classical_model.predict(X[val_idx])
classical_test_pred = classical_model.predict(X[test_idx])
classical_val_proba = classical_model.predict_proba(X[val_idx])
classical_test_proba = classical_model.predict_proba(X[test_idx])
all_classical_pred = classical_model.predict(X)
all_classical_proba = classical_model.predict_proba(X)

classical_val_metrics = compute_metrics(y[val_idx], classical_val_pred, "Classical baseline", "validation")
classical_test_metrics = compute_metrics(y[test_idx], classical_test_pred, "Classical baseline", "test")
classical_cm = confusion_matrix(y[test_idx], classical_test_pred)

meta_df["classical_pred_id"] = all_classical_pred
meta_df["classical_pred"] = meta_df["classical_pred_id"].map(id_to_display)
meta_df["classical_confidence"] = all_classical_proba.max(axis=1)

print(f"Train size: {len(train_idx)} | Validation size: {len(val_idx)} | Test size: {len(test_idx)}")
print("Classical baseline trained successfully.")
display(pd.DataFrame([classical_val_metrics, classical_test_metrics]).round(3))
print("Classical test-set classification report:")
print(classification_report(y[test_idx], classical_test_pred, target_names=class_names, zero_division=0))

In [ ]:
print("Visualising the engineered audio feature space and the classical model results...")

scaled_X = StandardScaler().fit_transform(X)
pca_projection = PCA(n_components=2, random_state=SEED).fit_transform(scaled_X)
pca_df = meta_df.copy()
pca_df["pc1"] = pca_projection[:, 0]
pca_df["pc2"] = pca_projection[:, 1]

fig_pca = px.scatter(
    pca_df,
    x="pc1",
    y="pc2",
    color="label_name",
    symbol="split",
    hover_data=["sample_id", "description"],
    title="Engineered audio features projected into 2D (PCA)",
    template="plotly_white",
)
fig_pca.show(renderer="notebook")

classical_cm_df = pd.DataFrame(
    classical_cm,
    index=[f"True: {name}" for name in class_names],
    columns=[f"Pred: {name}" for name in class_names],
)

fig_classical_cm = px.imshow(
    classical_cm_df,
    text_auto=True,
    color_continuous_scale="Blues",
    aspect="auto",
    title="Classical baseline confusion matrix (test set)",
)
fig_classical_cm.update_layout(coloraxis_showscale=False, template="plotly_white")
fig_classical_cm.show(renderer="notebook")

## 6. A Simple Neural Audio Model with PyTorch

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Train a neural network that works directly on mel-spectrograms. This lets the model learn patterns from a more structured audio representation instead of relying only on hand-crafted summary features.
</div>

We will keep the architecture intentionally small and classroom-friendly:
- mel-spectrogram input
- a compact convolutional neural network
- validation tracking during training
- final test-set evaluation

This makes the workflow modern and realistic without becoming too heavy for CPU-based notebook use.

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Exercise:</b> Later, try changing <code>EPOCHS</code> or <code>BATCH_SIZE</code> in the training cell. This is a safe way to explore how neural training behaviour changes.
</div>

In [ ]:
print("Preparing spectrogram tensors and training a neural audio model...")

all_specs = []
for waveform, sample_rate in zip(waveforms, meta_df["sample_rate"]):
    mel_db = waveform_to_mel_db(waveform, int(sample_rate))
    mel_db = pad_or_trim_spec(mel_db, target_frames=TARGET_FRAMES)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
    all_specs.append(mel_db.unsqueeze(0))

all_specs = torch.stack(all_specs)
all_labels = torch.tensor(y, dtype=torch.long)

tensor_dataset = TensorDataset(all_specs, all_labels)

# EDIT THIS VALUE: batch size for neural training.
BATCH_SIZE = 16
# EDIT THIS VALUE: number of training epochs.
EPOCHS = 12
# EDIT THIS VALUE: learning rate for the optimiser.
LEARNING_RATE = 0.001

train_loader = DataLoader(Subset(tensor_dataset, train_idx.tolist()), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(Subset(tensor_dataset, val_idx.tolist()), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(Subset(tensor_dataset, test_idx.tolist()), batch_size=BATCH_SIZE, shuffle=False)
all_loader = DataLoader(tensor_dataset, batch_size=BATCH_SIZE, shuffle=False)

class SmallAudioCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(8),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(16),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

neural_model = SmallAudioCNN(num_classes=len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(neural_model.parameters(), lr=LEARNING_RATE)

def evaluate_torch_model(model, data_loader):
    model.eval()
    losses = []
    true_labels = []
    predictions = []
    probabilities = []
    with torch.no_grad():
        for xb, yb in data_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            losses.append(loss.item())
            true_labels.extend(yb.detach().cpu().numpy().tolist())
            predictions.extend(preds.detach().cpu().numpy().tolist())
            probabilities.extend(probs.detach().cpu().numpy().tolist())
    metrics = compute_metrics(true_labels, predictions, "Neural CNN", "loader")
    return float(np.mean(losses)), metrics, np.array(probabilities), np.array(predictions), np.array(true_labels)

history = []
best_state = copy.deepcopy(neural_model.state_dict())
best_val_f1 = -1.0

print("Training neural model...")
for epoch in range(1, EPOCHS + 1):
    neural_model.train()
    train_losses = []
    train_true = []
    train_pred = []

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = neural_model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_true.extend(yb.detach().cpu().numpy().tolist())
        train_pred.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())

    train_metrics = compute_metrics(train_true, train_pred, "Neural CNN", "train")
    val_loss, val_metrics, _, _, _ = evaluate_torch_model(neural_model, val_loader)
    val_metrics["split"] = "validation"

    history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "train_accuracy": train_metrics["accuracy"],
            "train_f1": train_metrics["f1"],
            "val_loss": val_loss,
            "val_accuracy": val_metrics["accuracy"],
            "val_f1": val_metrics["f1"],
        }
    )

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_state = copy.deepcopy(neural_model.state_dict())

    print(
        f"Epoch {epoch}/{EPOCHS} | train loss: {np.mean(train_losses):.4f} | train acc: {train_metrics['accuracy']:.3f} | val acc: {val_metrics['accuracy']:.3f} | val f1: {val_metrics['f1']:.3f}"
    )

neural_model.load_state_dict(best_state)

neural_val_loss, neural_val_metrics, _, _, _ = evaluate_torch_model(neural_model, val_loader)
neural_test_loss, neural_test_metrics, neural_test_proba, neural_test_pred, neural_test_true = evaluate_torch_model(neural_model, test_loader)
_, _, all_neural_proba, all_neural_pred, _ = evaluate_torch_model(neural_model, all_loader)

neural_val_metrics["split"] = "validation"
neural_test_metrics["split"] = "test"
neural_cm = confusion_matrix(neural_test_true, neural_test_pred)

meta_df["neural_pred_id"] = all_neural_pred
meta_df["neural_pred"] = meta_df["neural_pred_id"].map(id_to_display)
meta_df["neural_confidence"] = all_neural_proba.max(axis=1)

print("Neural audio model training complete.")
display(pd.DataFrame([neural_val_metrics, neural_test_metrics]).round(3))
print("Neural test-set classification report:")
print(classification_report(neural_test_true, neural_test_pred, target_names=class_names, zero_division=0))

In [ ]:
print("Visualising neural training history and neural test results...")

history_df = pd.DataFrame(history)

fig_history = go.Figure()
fig_history.add_trace(go.Scatter(x=history_df["epoch"], y=history_df["train_accuracy"], mode="lines+markers", name="Train accuracy"))
fig_history.add_trace(go.Scatter(x=history_df["epoch"], y=history_df["val_accuracy"], mode="lines+markers", name="Validation accuracy"))
fig_history.add_trace(go.Scatter(x=history_df["epoch"], y=history_df["train_f1"], mode="lines+markers", name="Train F1"))
fig_history.add_trace(go.Scatter(x=history_df["epoch"], y=history_df["val_f1"], mode="lines+markers", name="Validation F1"))
fig_history.update_layout(
    title="Neural model learning curve",
    xaxis_title="Epoch",
    yaxis_title="Score",
    yaxis=dict(range=[0, 1]),
    template="plotly_white",
)
fig_history.show(renderer="notebook")

neural_cm_df = pd.DataFrame(
    neural_cm,
    index=[f"True: {name}" for name in class_names],
    columns=[f"Pred: {name}" for name in class_names],
)

fig_neural_cm = px.imshow(
    neural_cm_df,
    text_auto=True,
    color_continuous_scale="Greens",
    aspect="auto",
    title="Neural model confusion matrix (test set)",
)
fig_neural_cm.update_layout(coloraxis_showscale=False, template="plotly_white")
fig_neural_cm.show(renderer="notebook")

## 7. Comparing the Classical and Neural Approaches

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Key idea:</b> The point is not just to train a neural model because it feels modern. The point is to compare models carefully and learn what each approach captures well.
</div>

In some tasks, a well-designed classical model can be surprisingly strong. In other tasks, the neural model benefits from learning richer representations directly from structured inputs like spectrograms.

In [ ]:
print("Comparing baseline and neural performance side by side...")

comparison_df = pd.DataFrame(
    [
        {
            "model": "Classical engineered features + Logistic Regression",
            "accuracy": classical_test_metrics["accuracy"],
            "precision": classical_test_metrics["precision"],
            "recall": classical_test_metrics["recall"],
            "f1": classical_test_metrics["f1"],
        },
        {
            "model": "Neural mel-spectrogram CNN",
            "accuracy": neural_test_metrics["accuracy"],
            "precision": neural_test_metrics["precision"],
            "recall": neural_test_metrics["recall"],
            "f1": neural_test_metrics["f1"],
        },
    ]
).round(3)

display(comparison_df)

comparison_long = comparison_df.melt(id_vars="model", var_name="metric", value_name="score")
fig_compare = px.bar(
    comparison_long,
    x="metric",
    y="score",
    color="model",
    barmode="group",
    text="score",
    title="Test-set comparison: classical baseline vs neural model",
    template="plotly_white",
)
fig_compare.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig_compare.update_layout(yaxis=dict(range=[0, 1]))
fig_compare.show()

test_results_df = meta_df.iloc[test_idx].copy()
test_results_df["models_disagree"] = test_results_df["classical_pred_id"] != test_results_df["neural_pred_id"]
test_results_df["either_wrong"] = (
    (test_results_df["classical_pred_id"] != test_results_df["label_id"]) |
    (test_results_df["neural_pred_id"] != test_results_df["label_id"])
)

interesting_examples = test_results_df.loc[
    test_results_df["models_disagree"] | test_results_df["either_wrong"],
    [
        "sample_id",
        "label_name",
        "classical_pred",
        "classical_confidence",
        "neural_pred",
        "neural_confidence",
        "description",
    ],
].head(8)

print(f"Test examples where the models disagree: {int(test_results_df['models_disagree'].sum())}")
print("A few challenging test clips:")
display(interesting_examples)

## 8. Interactive Audio Explorer

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Try it yourself:</b> Use the explorer below to browse clips, listen to the audio, inspect the waveform and spectrogram, and compare both models' predicted probabilities.
</div>

This is where the notebook becomes especially powerful: you can connect the **sound**, the **visual representation**, and the **model behaviour** all in one place.

In [ ]:
print("Building the interactive audio explorer...")

sample_options = [
    (
        f"Sample {int(row['sample_id'])} | {row['label_short']} | {row['split']}",
        int(index),
    )
    for index, row in meta_df.iterrows()
]

sample_slider = widgets.SelectionSlider(
    options=sample_options,
    description="Clip:",
    continuous_update=False,
    layout=widgets.Layout(width="95%"),
)
explorer_output = widgets.Output()

def render_sample(sample_index):
    row = meta_df.iloc[sample_index]
    waveform = waveforms[sample_index]
    sample_rate = int(row["sample_rate"])
    mel_db = waveform_to_mel_db(waveform, sample_rate)

    info_df = pd.DataFrame(
        [
            {
                "sample_id": int(row["sample_id"]),
                "split": row["split"],
                "true_label": row["label_name"],
                "classical_prediction": f"{row['classical_pred']} ({row['classical_confidence']:.3f})",
                "neural_prediction": f"{row['neural_pred']} ({row['neural_confidence']:.3f})",
                "duration_sec": round(float(row["duration_sec"]), 3),
            }
        ]
    )

    probability_df = pd.DataFrame(
        {
            "class_name": class_names + class_names,
            "probability": list(all_classical_proba[sample_index]) + list(all_neural_proba[sample_index]),
            "model": ["Classical baseline"] * len(class_names) + ["Neural CNN"] * len(class_names),
        }
    )

    with explorer_output:
        clear_output(wait=True)
        print(f"Now exploring sample {int(row['sample_id'])} from the {row['split']} split.")
        print(f"Metadata: {row['description']}")
        display(info_df)
        display(Audio(waveform.detach().cpu().numpy(), rate=sample_rate))
        make_waveform_figure(waveform, sample_rate, f"Waveform | sample {int(row['sample_id'])}").show(renderer="notebook")
        make_spectrogram_figure(mel_db, f"Mel-spectrogram | sample {int(row['sample_id'])}").show(renderer="notebook")

        fig_probs = px.bar(
            probability_df,
            x="class_name",
            y="probability",
            color="model",
            barmode="group",
            text="probability",
            title="Predicted class probabilities",
            template="plotly_white",
        )
        fig_probs.update_traces(texttemplate="%{text:.3f}", textposition="outside")
        fig_probs.update_layout(yaxis=dict(range=[0, 1]), xaxis_title="Class", yaxis_title="Probability")
        fig_probs.show(renderer="notebook")

def on_sample_change(change):
    if change["name"] == "value":
        render_sample(change["new"])

sample_slider.observe(on_sample_change, names="value")

display(widgets.VBox([sample_slider, explorer_output]))
render_sample(sample_slider.value)
print("Interactive explorer ready! Move the slider to inspect different clips.")
print("Running this cell will have problems displaying the graphs for the first time only, just move the slider around to fix it.")

## 9. Limitations, Edge Cases, and Responsible Use

<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Be careful:</b> Strong notebook results do not mean we have solved audio understanding in general. Audio models can be sensitive to noise, recording quality, accent, microphone differences, and dataset size.
</div>

Important limitations to keep in mind:

- **Small dataset:** This notebook is designed for learning, not for claiming state-of-the-art results.
- **Task simplicity:** Binary tasks are useful for teaching, but real audio intelligence often involves messier labels and more ambiguity.
- **Recording conditions matter:** A model may behave differently when audio is louder, quieter, more echoic, or more compressed.
- **Confidence is not certainty:** A high score can still be wrong.
- **Responsible deployment:** Real audio systems can affect accessibility, privacy, and fairness. They should be tested carefully in the settings where they will actually be used.

A good real-world workflow would include:
- more varied training data
- stronger validation on unseen conditions
- careful error inspection
- documentation of limitations before deployment

## Take-away

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    Congratulations! You have built an end-to-end audio machine learning workflow. You explored waveform and spectrogram views, extracted compact audio features for a classical baseline, trained a neural PyTorch model on mel-spectrograms, compared both approaches carefully, and inspected predictions interactively.
</div>

### Suggested next steps
- Try different values for <code>N_MELS</code>, <code>TARGET_FRAMES</code>, or <code>EPOCHS</code>
- Inspect more clips in the interactive explorer and look for failure patterns
- Swap the neural model for a deeper CNN or a small recurrent model
- Extend the notebook to a multi-class audio problem
- Compare mel-spectrograms with other feature designs such as MFCC-style summaries


![Noteable license](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/Images/Noteable%20Notebook%20Footer.png)